[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-movierecommender.ipynb)

# Full Project: Movie Recommender System (Collaborative Filtering)

*AIBits Academy · Machine Learning End To End · Full Project*

100,000 real ratings, three prediction strategies of increasing sophistication, and the technique family — collaborative filtering — that none of this course's other 18 projects touch.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Packages that Colab does not ship by default (a no-op if already installed)
%pip install -q scikit-surprise

# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

def fetch(url, target, member=None):   # public source; a zip member is extracted and renamed to `target`
    if os.path.exists(target):
        return
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    blob = urllib.request.urlopen(req, timeout=120).read()
    if member:
        blob = zipfile.ZipFile(io.BytesIO(blob)).read(member)
    open(target, 'wb').write(blob)
    print('downloaded', target)

fetch('https://files.grouplens.org/datasets/movielens/ml-100k.zip', 'u.data', 'ml-100k/u.data')

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> An Indian OTT platform (Hotstar/Netflix-style) wants a "you might also like" engine: given a user's past ratings and nothing else about the movies themselves (no genre tags, no cast, no plot — purely the rating patterns), predict how they'd rate movies they haven't seen yet, well enough to recommend the ones they'd rate highest. This is the classic **collaborative filtering** problem — recommending purely from the wisdom of the crowd's rating behaviour, which is exactly what the course's chapter introduces conceptually. This project is its full, end-to-end companion.

> **Dataset**
>
> **MovieLens 100K** — 100,000 ratings (1–5 stars) from 943 users on 1,682 movies, the standard public benchmark for recommender research. [Dataset source (GroupLens) →](https://grouplens.org/datasets/movielens/100k/)

## Step 1 — Load and Split

An 80/20 random split of the 100,000 individual ratings (not users or movies) means every method below is evaluated on genuinely unseen (user, movie) pairs — some involving users or movies the model has partially seen elsewhere, exactly like a real production recommender:

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

ratings = pd.read_csv('u.data', sep='\t', names=['user','item','rating','ts'])
print(f"Ratings: {len(ratings)}   Users: {ratings.user.nunique()}   Movies: {ratings.item.nunique()}")

train, test = train_test_split(ratings, test_size=0.2, random_state=42)
print(f"Train: {len(train)}   Test: {len(test)}")

## Step 2 — Baseline: The Global Mean

Before any modelling, the simplest possible predictor — guess the average rating for every single (user, movie) pair — sets the floor every real method must beat:

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

global_mean = train.rating.mean()
baseline_preds = np.full(len(test), global_mean)
baseline_rmse = np.sqrt(mean_squared_error(test.rating, baseline_preds))
print(f"Global mean rating: {global_mean:.4f}")
print(f"Baseline RMSE: {baseline_rmse:.4f}")

## Step 3 — Item-Based Collaborative Filtering

For each (user, movie) test pair, look at every *other* movie that user rated in the training set, weight each by its cosine similarity (computed from train co-ratings) to the target movie, and predict a similarity-weighted average — "users who liked what you liked also liked…":

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# Sparse user x item train matrix, item-item cosine similarity
user_item = train.pivot_table(index='user', columns='item', values='rating').fillna(0)
item_sim = cosine_similarity(csr_matrix(user_item.T.values))

def predict_item_cf(user, item, k=30):
    if user not in user_item.index or item not in user_item.columns:
        return global_mean
    rated = user_item.loc[user][user_item.loc[user] > 0]
    sims = item_sim[user_item.columns.get_loc(item), [user_item.columns.get_loc(j) for j in rated.index]]
    top_k = np.argsort(sims)[-k:]
    if sims[top_k].sum() == 0: return global_mean
    return np.clip(np.dot(sims[top_k], rated.values[top_k]) / sims[top_k].sum(), 1, 5)

item_cf_preds = [predict_item_cf(u, i) for u, i in zip(test.user, test.item)]
print(f"Item-based CF RMSE: {np.sqrt(mean_squared_error(test.rating, item_cf_preds)):.4f}")

A meaningful improvement over the 1.1204 baseline — using each user's own other ratings, weighted by which movies genuinely rate similarly across the whole community, already captures real signal the global average cannot.

## Step 4 — Matrix Factorization (SVD-Style, Learned via SGD)

Item-based CF only looks at pairwise item similarity. Matrix factorization instead learns a small set of **latent factors** per user and per movie (unlabelled dimensions the model discovers on its own — not "genre" or "runtime" explicitly, but whatever combination of hidden factors best explains the ratings), then predicts a rating as the dot product of a user's and a movie's factor vectors, exactly the Netflix Prize-winning approach:

In [ ]:
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split as surprise_split

reader = Reader(rating_scale=(1,5))
data = Dataset.load_from_df(ratings[['user','item','rating']], reader)
trainset, testset = surprise_split(data, test_size=0.2, random_state=42)

svd = SVD(n_factors=20, n_epochs=30, lr_all=0.005, reg_all=0.02, random_state=42)
svd.fit(trainset)
preds = svd.test(testset)
print(f"Matrix Factorization RMSE: {accuracy.rmse(preds, verbose=False):.4f}")

## Comparing All Three

Each step adds real predictive signal: item-based CF beats the baseline by using *which* movies rate similarly; matrix factorization beats item-based CF by learning latent structure across the *entire* rating matrix at once, rather than comparing movies pairwise. This monotonic improvement (1.1204 → 0.9792 → 0.9230) is the same "more of the data's structure used, better result" pattern seen throughout this course — here specifically because matrix factorization can borrow strength from a user's ratings on movies with no direct similarity path to the target movie, something item-based CF structurally cannot do.

> **💡 Cold Start — the One Problem Neither Method Solves**
>
> Both methods fall back to the global mean whenever a user or movie has zero training ratings (see the `if user not in ... return global_mean` line in Step 3) — a brand-new user or a brand-new movie gets no personalisation at all until they accumulate a handful of ratings. This is the well-known **cold-start problem** in collaborative filtering, and it's precisely where *content-based* features (genre, cast, director — the metadata this project deliberately excluded to isolate pure collaborative filtering) would need to supplement the rating-only approach in a production system, typically via a hybrid recommender.

## Key Business Takeaways

- Collaborative filtering needs no information about the movies themselves — no genre, no cast, no plot — only the pattern of who rated what. That's both its strength (works for any item catalogue) and its weakness (cold start).
- Matrix factorization (RMSE 0.9230) beats item-based CF (RMSE 0.9792) by learning latent structure across the whole rating matrix at once, rather than one pairwise similarity at a time — a 5.7% RMSE reduction over item-based CF, and 17.6% over the naive global-mean baseline.
- Both real methods leave 20–30% of the achievable RMSE reduction on the table relative to state-of-the-art (deep, hybrid) recommenders — but both are simple, fast, and interpretable enough to ship as a first production version.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · How sparse is the rating matrix?

Store in `sparsity` the fraction of (user, movie) cells that have **no** rating: `1 - n_ratings / (n_users * n_movies)`.

In [ ]:
sparsity = None   # TODO (use `ratings`)


In [ ]:
try:
    check("about 93.7% empty", abs(sparsity - (1 - 100000 / (943 * 1682))) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
sparsity = 1 - len(ratings) / (ratings.user.nunique() * ratings.item.nunique())

```

</details>

### Exercise 2 · Medium · A user-mean baseline

Predict each test rating by the **user's own average training rating** (fall back to `global_mean` for unseen users). Store the RMSE in `rmse_user_mean`; it should beat the global-mean baseline.

In [ ]:
rmse_user_mean = None   # TODO (use train, test, global_mean)


In [ ]:
try:
    check("beats the global-mean baseline", rmse_user_mean < baseline_rmse)
    check("about 1.04", 0.98 < rmse_user_mean < 1.10)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
user_mean = train.groupby("user")["rating"].mean()
pred = test["user"].map(user_mean).fillna(global_mean)
rmse_user_mean = float(np.sqrt(mean_squared_error(test["rating"], pred)))

```

</details>

### Exercise 3 · Stretch · Most-rated movies

Store in `top5` the ids of the five movies with the most ratings in `train` (most rated first), and in `top5_mean` their average rating (a float).

In [ ]:
top5 = top5_mean = None   # TODO


In [ ]:
try:
    ref = train["item"].value_counts().head(5).index.tolist()
    check("same movies", top5 == ref)
    check("mean of their ratings", abs(top5_mean - train[train["item"].isin(ref)]["rating"].mean()) < 1e-12)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
top5 = train["item"].value_counts().head(5).index.tolist()
top5_mean = float(train[train["item"].isin(top5)]["rating"].mean())

```

Popularity is itself a strong recommender baseline - any personalised model has to beat it.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Movie Recommender System (Collaborative Filtering)**.*